<a href="https://colab.research.google.com/github/gvgabison/Sample-LLM-ChatGPT/blob/main/RAGQwen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#Install and import the libraries needed for PDF reading, embeddings, vector search and Qwen 2.5 1.5B for  answer generation.
!pip -q install pymupdf faiss-cpu numpy sentence-transformers transformers accelerate torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 22.5 MB/s eta 0:00:00


In [2]:
import time
import re
from typing import List

import numpy as np
import faiss
import fitz  # PyMuPDF
import torch
from google.colab import files
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM

#This is the local embedding model used to convert text chunks and user questions into vectors.
#all-MiniLM-L6-v2 is commonly used for semantic search and retrieval tasks.
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

#This is the local LLM used for answer generation.
#Qwen/Qwen2.5-1.5B-Instruct is the instruction-tuned Qwen 2.5 1.5B model on Hugging Face.
GEN_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

#Load the embedding model once so it can be reused for chunking and retrieval.
embedder = SentenceTransformer(EMBED_MODEL)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [3]:
#Load the tokenizer and Qwen model for text generation.
#device_map='auto' helps Colab place the model on GPU if available.
tokenizer = AutoTokenizer.from_pretrained(GEN_MODEL)
model = AutoModelForCausalLM.from_pretrained(
    GEN_MODEL,
    dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [4]:
#Upload the PDF file that will serve as the knowledge source.
uploaded = files.upload()
pdf_file = list(uploaded.keys())[0]
print("Uploaded:", pdf_file)

#This function extracts text from every page of the PDF and combines it into one long string.
def extract_text_from_pdf(pdf_path: str) -> str:
    doc = fitz.open(pdf_path)
    pages = []
    for i in range(len(doc)):
        pages.append(doc.load_page(i).get_text())
    return "\n".join(pages)

raw_text = extract_text_from_pdf(pdf_file).strip()
print("Extracted characters:", len(raw_text))
print(raw_text[:600])


Saving HR-Manual.pdf to HR-Manual.pdf
Uploaded: HR-Manual.pdf
Extracted characters: 152333
Human Resources 
Policies and Procedures Manual 
October 2015 
 
 
 
 
 
 
 
 
 

 
1 
 
TABLE OF CONTENTS 
 
 
Page Number 
SECTION 1 
 
INTRODUCTORY 
TCCAP Organizational Chart ..................................................................................................... 1 
Policy and Procedure Manual  .................................................................................................... 2 
 
 
SECTION 2 
 
EMPLOYMENT POLICIES AND PRACTICES 
Employment-At-Will ................................................................................................................


In [5]:
#The helper function below splits text while preserving the separator.
def _split_by_separator(text: str, sep: str) -> List[str]:
    if sep == "":
        return list(text)
    parts = text.split(sep)
    out = []
    for i, p in enumerate(parts):
        out.append(p + sep if i < len(parts) - 1 else p)
    return out

#This recursive splitter breaks the text into smaller overlapping chunks.
#Chunk overlap helps preserve context between neighboring chunks.
def recursive_text_splitter(
    text: str,
    chunk_size: int = 900,
    chunk_overlap: int = 120,
    separators: List[str] = None,
    min_chunk_chars: int = 200
) -> List[str]:
    if separators is None:
        separators = ["\n\n", "\n", ". ", " ", ""]

    text = (text or "").strip()
    if not text:
        return []

    def _recurse(t: str, seps: List[str]) -> List[str]:
        if len(t) <= chunk_size or not seps:
            return [t]

        sep = seps[0]
        parts = _split_by_separator(t, sep)
        if len(parts) == 1:
            return _recurse(t, seps[1:])

        out, buf = [], ""
        for part in parts:
            if len(part) > chunk_size and seps[1:]:
                if buf.strip():
                    out.append(buf)
                    buf = ""
                out.extend(_recurse(part, seps[1:]))
                continue

            if len(buf) + len(part) <= chunk_size:
                buf += part
            else:
                if buf.strip():
                    out.append(buf)
                buf = part

        if buf.strip():
            out.append(buf)
        return out

    pieces = _recurse(text, separators)

    cleaned = []
    for p in pieces:
        p = p.strip()
        if not p:
            continue
        if cleaned and len(p) < min_chunk_chars:
            cleaned[-1] = (cleaned[-1].rstrip() + " " + p).strip()
        else:
            cleaned.append(p)

    chunks = []
    for p in cleaned:
        if not chunks:
            chunks.append(p)
            continue
        overlap_text = chunks[-1][-chunk_overlap:] if chunk_overlap > 0 else ""
        merged = (overlap_text + p).strip()
        if len(merged) > chunk_size:
            merged = merged[-chunk_size:]
        chunks.append(merged)

    return chunks

chunks = recursive_text_splitter(raw_text, chunk_size=900, chunk_overlap=120)
print("Chunks:", len(chunks))
print("Sample chunk:\n", chunks[0][:500])

Chunks: 196
Sample chunk:
 Human Resources 
Policies and Procedures Manual 
October 2015


In [6]:
#This function converts text into embeddings using the local sentence-transformer model.
def embed_texts(texts: List[str], batch_size: int = 64) -> np.ndarray:
    vectors = embedder.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=False,
        convert_to_numpy=True,
        normalize_embeddings=True
    )
    return np.array(vectors, dtype=np.float32)

#Generate embeddings for all chunks, then store them in FAISS for fast similarity search.
t0 = time.time()
chunk_vecs = embed_texts(chunks, batch_size=64)
t1 = time.time()

dim = chunk_vecs.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(chunk_vecs)

print("Embeddings shape:", chunk_vecs.shape)
print("FAISS vectors:", index.ntotal)
print("Embedding time (s):", round(t1 - t0, 2))

Embeddings shape: (196, 384)
FAISS vectors: 196
Embedding time (s): 46.64


In [7]:
# When a user asks a question, embed the question and search for the top matching chunks.
def retrieve(question: str, top_k: int = 3):
    q_vec = embed_texts([question], batch_size=1)
    scores, indices = index.search(q_vec, top_k)
    hits = []
    for rank, (idx, score) in enumerate(zip(indices[0], scores[0]), start=1):
        hits.append({"rank": rank, "idx": int(idx), "score": float(score), "text": chunks[int(idx)]})
    return hits

#This function sends only the retrieved chunks to Qwen so the answer stays grounded in the PDF.
def ask_question(question: str, top_k: int = 3) -> str:
    hits = retrieve(question, top_k=top_k)

    context_blocks = []
    for h in hits:
        context_blocks.append(f"[Source {h['rank']}]\n{h['text']}")

    context = "\n\n".join(context_blocks)

    prompt = (
        "You are a helpful assistant.\n"
        "Use ONLY the context to answer.\n"
        "If the answer is not in the context, say: I do not know.\n"
        "Write a clean answer in 1 to 3 sentences.\n"
        "Do not include numbering or bullet points.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {question}\n"
        "Answer:"
    )

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=160,
            do_sample=False,
            temperature=None,
            top_p=None,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = outputs[0][inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(generated, skip_special_tokens=True).strip()

    answer = re.sub(r"^Answer:\s*", "", answer).strip()
    answer = re.sub(r"^\s*\(?\d+\)?[.)]\s+", "", answer).strip()
    return answer

In [8]:
# This loop is for the interactive PDF question-answering chatbot.
while True:
    query = input("\nAsk a question (type 'exit' to stop): ").strip()
    if query.lower() == "exit":
        break

    print("\nAnswer:\n")
    print(ask_question(query, top_k=3))


Ask a question (type 'exit' to stop): what is a substitute


The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



Answer:

A substitute is an individual who is hired either full-time or part-time for a limited period (120 days) under certain conditions. They can be requested by the immediate supervisor and provide assistance with specific functions. Substitutes are typically used to cover temporary absences or to assist during peak periods when there's an increased workload. The context mentions they can replace teachers and teacher assistants, among other roles. However, it does not specify whether substitutes are permanent positions or temporary assignments.

Ask a question (type 'exit' to stop): who is the president

Answer:

I do not know.

Ask a question (type 'exit' to stop): onboarding

Answer:

I do not know.

Ask a question (type 'exit' to stop): exit
